# 🤖 Hibrit (Topluluk / Ensemble) Model Değerlendirmesi

Bu aşamada, klasik makine öğrenmesi modeli (LightGBM/SVC) ile derin öğrenme modelinin (BERT) güçlerini birleştirdiğimiz **Hibrit Model**in başarısını ölçeceğiz.

Modelimiz (Soft-Voting mantığıyla) her iki modelin de yüzdelik olasılıklarını alıp, onları toplayarak nihai kararı vermektedir.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from hybrid_model import HybridEnsembleClassifier

## 1. Verinin ve Modelin Yüklenmesi

In [ ]:
# Aynı test setini (random_state=42) tekrar ayırıyoruz
df = pd.read_csv('data/processed/reviews_cleaned.csv')
df = df.dropna(subset=['yorum'])

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
print(f"Test Seti Boyutu: {len(test_df)}")

# Hibrit Modeli Başlat
hybrid_model = HybridEnsembleClassifier()

## 2. Test Seti Üzerinde Tahminlerin Alınması

Bu işlem GPU üzerinde yaklaşık 1-2 dakika sürebilir.

In [ ]:
predictions = []
y_true = test_df['label'].tolist()
texts = test_df['yorum'].tolist()

for text in tqdm(texts, desc="Hibrit Model İle Tahmin Ediliyor"):
    label, _ = hybrid_model.predict(text)
    predictions.append(label)


## 3. Sonuçlar ve Karşılaştırma

In [ ]:
print("=======================================================")
print("        HİBRİT MODEL TEST SONUÇLARI")
print("=======================================================")
print(classification_report(y_true, predictions))

hybrid_f1 = f1_score(y_true, predictions, average='weighted')
hybrid_acc = accuracy_score(y_true, predictions)

print(f"Hibrit F1 Score: {hybrid_f1:.4f}")
print(f"Hibrit Accuracy: {hybrid_acc:.4f}")

In [ ]:
# Önceki sonuçlarla (Hardcoded) görselleştirme
models = ['Klasik (SVC/LGBM)', 'Sadece BERT', 'Hibrit (BERT + Klasik)']
f1_scores = [0.71, 0.64, hybrid_f1]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=f1_scores, palette='viridis')
plt.title('Modellerin F1 Skor Karşılaştırması', fontsize=14)
plt.ylabel('F1 Score')
plt.ylim(0, 1.0)

for i, v in enumerate(f1_scores):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')

plt.show()